# Week 3 — Baseline Model (BLS + Classical ML)
Continues from your Week 1/2 notebook. Reads `balanced_tce.csv` and the `.npy` files already saved to Google Drive — **no lightkurve / archive calls in this notebook.**

**Models trained:**
1. `BLS_baseline` — Logistic Regression on BLS-derived catalog stats only (`tce_period`, `tce_duration`, `tce_depth`, `tce_model_snr`)
2. `RF_main` — Random Forest on BLS stats + flux-shape features engineered from the `.npy` arrays (no `robovetter_score`)
3. `RF_with_robovetter` — same as above + `robovetter_score`, logged separately as an upper-bound / expert-prior reference, not part of the main comparison

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import numpy as np
import pandas as pd

DRIVE_ROOT = '/content/drive/MyDrive/SADA'
CSV_PATH   = os.path.join(DRIVE_ROOT, 'balanced_tce.csv')
NPY_DIR    = os.path.join(DRIVE_ROOT, 'processed')

assert os.path.exists(CSV_PATH), f"Can't find {CSV_PATH} — check the exact filename/path in Drive"
assert os.path.exists(NPY_DIR), f"Can't find {NPY_DIR}"
print('CSV found:', CSV_PATH)
print('NPY dir found:', NPY_DIR)

## 1. Load catalog + inventory the saved flux arrays
Filenames follow `{kepid}_{tce_plnt_num}_{PC|FP}.npy` (from Week 2). We parse that back out instead of re-deriving anything from Kepler.

In [ ]:
balanced = pd.read_csv(CSV_PATH)
print(f"balanced_tce.csv: {len(balanced)} rows")
print(balanced[['kepid','tce_plnt_num','disposition','tce_period','tce_duration',
                 'tce_depth','tce_model_snr','robovetter_score']].head())

npy_files = [f for f in os.listdir(NPY_DIR) if f.endswith('.npy') and f != 'failed_kepids.npy']
print(f"\n.npy files found: {len(npy_files)}")

pattern = re.compile(r'^(\d+)_(\d+)_(PC|FP)\.npy$')
records = []
for fname in npy_files:
    m = pattern.match(fname)
    if not m:
        continue
    kepid, plnt_num, label_str = m.groups()
    records.append({
        'kepid': int(kepid),
        'tce_plnt_num': int(plnt_num),
        'label': 1 if label_str == 'PC' else 0,
        'npy_path': os.path.join(NPY_DIR, fname)
    })

npy_index = pd.DataFrame(records)
print(f"Parsed {len(npy_index)} filenames into an index")

## 2. Join flux-array index to the catalog
This gives every row both the BLS catalog stats *and* a path to its folded flux array — purely a local merge, nothing fetched.

In [ ]:
data = pd.merge(
    npy_index,
    balanced,
    on=['kepid', 'tce_plnt_num'],
    how='inner'
)

# sanity check: label parsed from filename should match disposition column
mismatch = data[(data['label'] == 1) != (data['disposition'] == 'PC')]
print(f"Rows: {len(data)} | label/disposition mismatches: {len(mismatch)}")

data = data.dropna(subset=['tce_period','tce_duration','tce_depth','tce_model_snr']).reset_index(drop=True)
print(f"After dropping rows with missing BLS stats: {len(data)}")
print(f"PC: {(data['label']==1).sum()} | FP: {(data['label']==0).sum()}")

## 3. Engineer flux-shape features from the folded arrays
Cheap, local, no fetching — just numpy over the 200-point arrays you already have.

In [ ]:
def flux_shape_features(flux):
    flux = np.asarray(flux, dtype=float)
    n = len(flux)
    mean = flux.mean()
    std = flux.std()
    minimum = flux.min()
    maximum = flux.max()

    # crude in-transit window: the central third, where a folded transit dip should sit
    lo, hi = n // 3, 2 * n // 3
    in_transit = flux[lo:hi]
    out_transit = np.concatenate([flux[:lo], flux[hi:]])

    depth_est = out_transit.mean() - in_transit.min()          # how deep the dip goes
    in_transit_std = in_transit.std()
    out_transit_std = out_transit.std() if len(out_transit) else 0.0

    # skew / kurtosis without scipy dependency
    if std > 0:
        skew = ((flux - mean) ** 3).mean() / (std ** 3)
        kurt = ((flux - mean) ** 4).mean() / (std ** 4) - 3
    else:
        skew, kurt = 0.0, 0.0

    return {
        'flux_mean': mean,
        'flux_std': std,
        'flux_min': minimum,
        'flux_max': maximum,
        'flux_depth_est': depth_est,
        'flux_in_transit_std': in_transit_std,
        'flux_out_transit_std': out_transit_std,
        'flux_skew': skew,
        'flux_kurtosis': kurt,
    }

feature_rows = []
for path in data['npy_path']:
    flux = np.load(path)
    feature_rows.append(flux_shape_features(flux))

flux_feats = pd.DataFrame(feature_rows)
data = pd.concat([data.reset_index(drop=True), flux_feats], axis=1)
print(f"Engineered {flux_feats.shape[1]} flux-shape features")
data[flux_feats.columns].describe().T

## 4. Define feature sets for each model
`robovetter_score` is only used in the separate upper-bound run.

In [ ]:
BLS_FEATURES = ['tce_period', 'tce_duration', 'tce_depth', 'tce_model_snr']

FLUX_FEATURES = ['flux_mean', 'flux_std', 'flux_min', 'flux_max', 'flux_depth_est',
                 'flux_in_transit_std', 'flux_out_transit_std', 'flux_skew', 'flux_kurtosis']

RF_FEATURES = BLS_FEATURES + FLUX_FEATURES                      # main baseline, no robovetter
RF_FEATURES_WITH_ROBOVETTER = RF_FEATURES + ['robovetter_score'] # upper-bound reference

print('BLS baseline features:', BLS_FEATURES)
print('RF main features:', RF_FEATURES)
print('RF upper-bound features:', RF_FEATURES_WITH_ROBOVETTER)

## 5. Train / val / test split
Same 70/15/15, `random_state=42` split logic as your CNN dataloader (Week 2, cell 19) so the three model families stay comparable on identical rows.

In [ ]:
from sklearn.model_selection import train_test_split

y = data['label'].values

idx_train, idx_temp = train_test_split(
    data.index, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])

train_df, val_df, test_df = data.loc[idx_train], data.loc[idx_val], data.loc[idx_test]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 6. Train the three models + shared evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

results = []

def evaluate(name, model, feature_cols, scale=False):
    X_train = train_df[feature_cols].values
    X_val   = val_df[feature_cols].values
    X_test  = test_df[feature_cols].values
    y_train, y_val, y_test = train_df['label'].values, val_df['label'].values, test_df['label'].values

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train, X_val, X_test = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    row = {
        'model': name,
        'n_features': len(feature_cols),
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'roc_auc': roc_auc_score(y_test, probs),
    }
    results.append(row)
    print(f"\n=== {name} ===")
    for k, v in row.items():
        if k not in ('model',):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print('  confusion matrix:\n', confusion_matrix(y_test, preds))
    return model

# 1. BLS baseline
bls_model = evaluate(
    'BLS_baseline',
    LogisticRegression(max_iter=1000),
    BLS_FEATURES,
    scale=True
)

# 2. RF main (no robovetter)
rf_main = evaluate(
    'RF_main',
    RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    RF_FEATURES
)

# 3. RF with robovetter — separate upper-bound reference, not part of main comparison
rf_upper = evaluate(
    'RF_with_robovetter (upper bound / expert prior)',
    RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    RF_FEATURES_WITH_ROBOVETTER
)

results_df = pd.DataFrame(results)
results_df

## 7. Feature importances (RF_main) — sanity check that the flux features are pulling weight

In [ ]:
importances = pd.Series(rf_main.feature_importances_, index=RF_FEATURES).sort_values(ascending=False)
importances.plot(kind='barh', figsize=(8,5), title='RF_main feature importances')
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()
importances

## 8. Save results log to Drive
Keeps a running record you can append to in Week 4 when the CNN numbers come in.

In [ ]:
RESULTS_PATH = os.path.join(DRIVE_ROOT, 'model_results.csv')

if os.path.exists(RESULTS_PATH):
    prior = pd.read_csv(RESULTS_PATH)
    combined = pd.concat([prior, results_df], ignore_index=True)
else:
    combined = results_df

combined.to_csv(RESULTS_PATH, index=False)
print(f"Saved results log to {RESULTS_PATH}")
combined